___
# <center>Regressão linear múltipla</center>
___

## Aula 19

**Objetivo da aula:** ao final desta aula, você deve ser capaz de:

 * interpretar o intercepto, o coeficiente angular, o intervalo e o $R^2$ de uma
   regressão a partir da saída do `statsmodels`;
 * interpretar cada coeficiente de uma regressão múltipla **mantidas as demais
   variáveis**, e usar a correlação entre as explicativas para entender por que
   ele muda quando uma variável entra ou sai;
 * reconhecer o teste t e a ANOVA como regressões com dummies, e ler cada
   coeficiente em relação à categoria de referência.

A pergunta da aula, do começo ao fim: **o que está associado ao valor pago pelo
banco nos processos em que ele pagou?**

___
<div id="indice"></div>

## Índice

- [Parte 1: a base](#base)
- [Parte 2: regressão simples e R²](#simples)
- [Parte 3: regressão múltipla](#multipla)
- [Parte 4: teste t e ANOVA como regressão](#dummy)
- [Exercícios em sala](#exercicios)

> **Sobre a base.** `btg_contratos.csv` é a planilha completa do projeto da clínica
> com o BTG (30 mil processos), com o registro público de parte dos processos no
> PDPJ (valor da causa, juizado ou vara, sentença), sem número de processo, de
> contrato nem dado pessoal. Está coberta por sigilo bancário e pelo termo de
> parceria: vem pelo BlackBoard e **não pode ser compartilhada** fora da turma, nem
> subida para o GitHub, nem colada em ferramentas externas. As bases dos
> exercícios 1 e 2 são públicas.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)

# Baixe btg_contratos.csv do BlackBoard e deixe na mesma pasta deste notebook.
# No Colab: clique no ícone de pasta, à esquerda, e arraste o arquivo para lá.
contratos = pd.read_csv("btg_contratos.csv")
contratos["causa_mil"] = contratos["valor_causa"] / 1000
contratos["financiado_mil"] = contratos["valor_financiado"] / 1000
contratos["anos_desde_2020"] = contratos["ano_ajuizamento"] - 2020
for col in ["juizado", "gratuidade", "teve_recurso"]:      # sim/não, vazio se não consultado
    contratos[col] = contratos[col].astype("boolean")
contratos.shape

[Volta ao Índice](#indice)

___
<div id="base"></div>

# Parte 1: a base

Uma linha por processo. As colunas do contrato vêm da planilha do banco; tribunal
e ano saem do número CNJ; valor da causa, juizado ou vara e sentença vêm do
registro público do processo (PDPJ); `pagou` e `valor_pago` dizem se o banco pagou
alguma coisa (acordo ou condenação), e quanto.

In [ ]:
contratos.head()

In [ ]:
print(f"processos com pagamento: {contratos['pagou'].mean():.1%}")
pagos = contratos[contratos["pagou"]].dropna(subset=["causa_mil"]).copy()
print(f"com pagamento e consultados no PDPJ: {len(pagos)}")
pagos[["valor_pago", "valor_causa", "valor_financiado"]].describe().round(0)

[Volta ao Índice](#indice)

___
<div id="simples"></div>

# Parte 2: regressão simples e R²

O valor pago contra o valor da causa (o que o autor pediu), em R$ mil.

In [ ]:
simples = smf.ols("valor_pago ~ ____", data=pagos).fit()
print(simples.summary().tables[1])
print(f"R² = {simples.rsquared:.3f}   n = {simples.nobs:.0f}")

* **Intercepto:** o valor pago previsto para uma causa de R$ 0. Fora dos dados,
  sem leitura prática.
* **Coeficiente:** quanto o valor pago muda, em média, a cada R$ 1.000 a mais no
  valor da causa, com o intervalo de 95% ao lado.
* **R²:** a fração da variação do valor pago que a reta explica.

**O R² por dentro.** A variação total é a soma dos quadrados das distâncias até a
média; o que sobra é a soma dos quadrados dos resíduos (distâncias até a reta).

In [ ]:
y = pagos["valor_pago"]
total = ((y - y.mean()) ** 2).sum()
sobra = (simples.____ ** 2).sum()          # os resíduos do modelo
print(f"R² na mão: {1 - sobra / total:.3f}")

r = pagos["causa_mil"].corr(pagos["valor_pago"])
print(f"correlação r = {r:.3f}, e r² = {r ** 2:.3f}")

Na regressão simples, $R^2 = r^2$. Com o **mesmo valor pago** e outra explicativa,
o valor financiado no contrato, o R² é menor: o pedido explica mais do que o
contrato.

In [ ]:
outra = smf.ols("valor_pago ~ financiado_mil", data=pagos).fit()
print(f"R² com o valor da causa: {simples.rsquared:.3f}; com o valor financiado: {outra.rsquared:.3f}")

[Volta ao Índice](#indice)

___
<div id="multipla"></div>

# Parte 3: regressão múltipla

Antes de pôr várias variáveis no modelo, a correlação entre elas.

In [ ]:
base_m = pagos.dropna(subset=["financiado_mil", "taxa_juros", "anos_desde_2020"])
cols = ["valor_pago", "causa_mil", "financiado_mil", "valor_parcela", "taxa_juros", "anos_desde_2020"]
base_m[cols].____().round(2)

A causa tem correlação com o financiado (causas maiores vêm de contratos
maiores), e o financiado tem correlação de 0,94 com a parcela (quase a mesma
informação). A função `compara` ajusta vários modelos e põe os coeficientes lado
a lado.

In [ ]:
def compara(formulas, dados):
    """Coeficientes e R² de vários modelos, um por coluna."""
    colunas = {}
    for f in formulas:
        m = smf.ols(f, data=dados).fit()
        colunas[f.split("~")[1].strip()] = pd.concat(
            [m.params, pd.Series({"R²": m.rsquared, "R² ajustado": m.rsquared_adj})])
    return pd.DataFrame(colunas).round(3)

compara(["valor_pago ~ causa_mil",
         "valor_pago ~ causa_mil + financiado_mil",
         "valor_pago ~ causa_mil + financiado_mil + taxa_juros + anos_desde_2020"], base_m)

O coeficiente da causa diminui quando o financiado entra: parte da associação da
causa com o valor pago vinha do tamanho do contrato. **Cada coeficiente vale
mantidas as demais variáveis do modelo.**

In [ ]:
completo = smf.ols("valor_pago ~ causa_mil + financiado_mil + taxa_juros + anos_desde_2020",
                   data=base_m).fit()
print(completo.summary().tables[1])

**Como relatar:** "Entre processos com o mesmo valor financiado, a mesma taxa de
juros e o mesmo ano, cada R$ 1.000 a mais no valor da causa estão associados a
cerca de R$ 44 a mais pagos", com o intervalo da saída.

[Volta ao Índice](#indice)

___
<div id="dummy"></div>

# Parte 4: teste t e ANOVA como regressão

**Teste t.** Juizado especial e vara comum: uma variável com duas categorias vira
uma dummy, e o coeficiente é a diferença de médias.

In [ ]:
from scipy import stats

t = pagos.dropna(subset=["juizado"]).copy()
t["onde"] = np.where(t["juizado"], "juizado especial", "vara comum")
print(t.groupby("onde")["valor_pago"].agg(["size", "mean"]).round(0))

m_t = smf.ols('valor_pago ~ C(onde, Treatment("____"))', data=t).fit()
print(m_t.summary().tables[1])
juiz = t["juizado"].astype(bool)
teste = stats.ttest_ind(t.loc[juiz, "valor_pago"], t.loc[~juiz, "valor_pago"])
print(f"teste t (variâncias iguais): t = {teste.statistic:.2f}")

**ANOVA.** O tribunal, com os seis maiores e os demais juntos: sete categorias,
seis dummies. A tabela da ANOVA testa se todas as diferenças são zero.

In [ ]:
principais = ["TJSP", "TJRS", "TJMG", "TJPR", "TJSC", "TJBA"]
pagos["trib"] = pagos["tribunal"].where(pagos["tribunal"].isin(principais), "outros")
print(pagos.groupby("trib")["valor_pago"].agg(["size", "mean"]).round(0))

m_trib = smf.ols('valor_pago ~ C(trib, Treatment("TJSP"))', data=pagos).fit()
print(m_trib.summary().tables[1])
print(sm.stats.anova_lm(m_trib, typ=2))

O intercepto é a média do TJSP, e cada coeficiente é a diferença da média do
tribunal para a do TJSP. Teste t, ANOVA e regressão são o mesmo modelo, com
variáveis explicativas de tipos diferentes.

[Volta ao Índice](#indice)

___
<div id="exercicios"></div>

# Exercícios em sala

Em duplas. Para cada saída, escreva as frases pedidas numa célula de texto logo
abaixo: **unidade**, **"mantidas as demais"** quando for múltipla, e **intervalo**
quando ele importar.

## Exercício 1: expectativa de vida nos países

Dados do Our World in Data para 2022: expectativa de vida (anos), PIB por
habitante (US$ mil), CO₂ por habitante (toneladas por ano) e população urbana (%).

In [ ]:
paises = pd.read_csv("paises_2022.csv")
paises.head()

In [ ]:
m1 = smf.ols("expectativa_vida ~ pib_per_capita_mil", data=paises).fit()
print(m1.summary().tables[1])
print(f"R² = {m1.rsquared:.3f}")

**a)** Interprete o coeficiente do PIB, com unidade. **b)** O intercepto tem
leitura prática aqui? **c)** O que o R² diz?

In [ ]:
m2 = smf.ols("expectativa_vida ~ co2_per_capita", data=paises).fit()
m3 = smf.ols("expectativa_vida ~ co2_per_capita + pib_per_capita_mil", data=paises).fit()
print(m2.summary().tables[1])
print(m3.summary().tables[1])
print(f"R²: só CO₂ {m2.rsquared:.3f}; CO₂ e PIB {m3.rsquared:.3f}")

**d)** O que aconteceu com o coeficiente do CO₂ quando o PIB entrou? E com o R²?
Use a correlação abaixo para explicar. **e)** Escreva a frase do coeficiente do
CO₂ no modelo com as duas variáveis.

In [ ]:
paises[["expectativa_vida", "pib_per_capita_mil", "co2_per_capita", "populacao_urbana_pct"]].corr().round(2)

## Exercício 2: casos novos nos tribunais estaduais

Os 27 tribunais de justiça em 2025: casos novos (em mil), população (em milhões)
e número de magistrados.

In [ ]:
tribunais = pd.read_csv("tribunais_2025.csv")
tribunais["casos_mil"] = tribunais["casos_novos"] / 1000
tribunais["pop_milhoes"] = tribunais["populacao"] / 1e6

print(tribunais[["casos_mil", "pop_milhoes", "magistrados"]].corr().round(3))
for f in ["casos_mil ~ pop_milhoes", "casos_mil ~ magistrados",
          "casos_mil ~ pop_milhoes + magistrados"]:
    m = smf.ols(f, data=tribunais).fit()
    print("\n", f, f"  R² = {m.rsquared:.3f}")
    print(m.summary().tables[1])

**a)** Interprete o coeficiente da população no primeiro modelo. **b)** Compare o
coeficiente e o erro-padrão (`std err`) da população no primeiro e no terceiro
modelo. O que mudou, e por quê? **c)** Faz sentido dizer, pelo terceiro modelo,
que "cada magistrado a mais traz mil casos novos"? O que a correlação de 0,98
entre população e magistrados permite e não permite concluir?

## Exercício 3: acrescentar e retirar variáveis no BTG

Partindo do modelo completo da parte 3, acrescente e retire variáveis e observe o
coeficiente da causa e o R².

In [ ]:
compara(["valor_pago ~ causa_mil + financiado_mil + taxa_juros + anos_desde_2020",
         "valor_pago ~ causa_mil + taxa_juros + anos_desde_2020",
         "valor_pago ~ causa_mil + financiado_mil + taxa_juros + anos_desde_2020 + juizado",
         "valor_pago ~ causa_mil + financiado_mil + taxa_juros + anos_desde_2020 + gratuidade"],
        base_m.dropna(subset=["juizado", "gratuidade"]).astype({"juizado": int, "gratuidade": int}))

**a)** Sem o financiado, o coeficiente da causa sobe ou desce? Confira na matriz
de correlação da parte 3 se era de esperar.

**b)** Com o juizado no modelo, o que muda no coeficiente da causa? E como você lê
o coeficiente do juizado, mantidas as demais?

**c)** Acrescente uma coluna de **puro ruído** e veja o que acontece com o R² e o
R² ajustado.

In [ ]:
rng = np.random.default_rng(19)
base_r = base_m.assign(ruido=rng.normal(size=len(base_m)))
compara(["valor_pago ~ causa_mil + financiado_mil + taxa_juros + anos_desde_2020",
         "valor_pago ~ causa_mil + financiado_mil + taxa_juros + anos_desde_2020 + ruido"], base_r)

## Exercício 4: duas variáveis quase iguais

O valor financiado e o valor da parcela têm correlação de 0,94. Compare o modelo
com uma só delas e o modelo com as duas.

In [ ]:
uma = smf.ols("valor_pago ~ causa_mil + financiado_mil", data=base_m).fit()
duas = smf.ols("valor_pago ~ causa_mil + financiado_mil + valor_parcela", data=base_m).fit()
print(uma.summary().tables[1])
print(duas.summary().tables[1])
print(f"R²: {uma.rsquared:.3f} e {duas.rsquared:.3f}")

**a)** O que aconteceu com o coeficiente e com o erro-padrão do financiado?
**b)** O R² mudou muito? **c)** Em uma frase: por que não se deve ler cada
coeficiente isoladamente quando as variáveis explicativas são quase a mesma
informação?

## Exercício 5: dummies e troca de referência

O valor pago por tipo de veículo, mantida a causa. Juntamos os tipos com poucos
processos em "outros".

In [ ]:
pagos["tipo"] = pagos["modalidade"].where(pagos["modalidade"].isin(["LEVES", "MOTO", "REVENDAS"]), "outros")
print(pagos.groupby("tipo")["valor_pago"].agg(["size", "mean"]).round(0))

m_leves = smf.ols('valor_pago ~ causa_mil + C(tipo, Treatment("LEVES"))', data=pagos).fit()
m_moto = smf.ols('valor_pago ~ causa_mil + C(tipo, Treatment("____"))', data=pagos).fit()
print(m_leves.summary().tables[1])
print(m_moto.summary().tables[1])

**a)** Com `LEVES` como referência, escreva a frase do coeficiente de `MOTO`,
mantido o valor da causa. **b)** Troque a referência para `MOTO`: o que mudou nos
coeficientes, e o que não mudou (compare o da causa e o R²)? **c)** A diferença
entre REVENDAS e LEVES é a mesma nos dois modelos?

## Exercício 6: a ANOVA da aula 17 como regressão

Na aula 17, comparamos a quantidade de pedidos da petição inicial entre as teses,
com a ANOVA. Refaça como regressão, com a amostra de 800 processos
(`btg_aulas.csv`, a mesma das aulas 16 a 18).

In [ ]:
processos = pd.read_csv("btg_aulas.csv")
ini = processos[processos["tem_inicial"]].dropna(subset=["qtd_pedidos"])
f = ini["tese_principal"].value_counts()
ini = ini[ini["tese_principal"].isin(f[f >= 15].index)]

reg_anova = smf.ols("qtd_pedidos ~ C(tese_principal)", data=ini).fit()
print(reg_anova.summary().tables[1])
print(sm.stats.anova_lm(reg_anova, typ=2).round(4))

**a)** Qual tese é a referência, e por quê? **b)** Interprete o coeficiente de
"outra". **c)** O F é o mesmo da aula 17 (7,26)? O que ele testa?

In [ ]:
# respostas e contas extras

[Volta ao Índice](#indice)